# A Deep Agent is just a regular agent with
- A small collection of essential tools
- A small collection of prebuilt middleware

### a few superpowers
- The ability to read from and write to filesystems
- The ability to spawn (sync/async) subagents

### and the ability to plan
- The builtin **todo list** middleware

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from util import pretty_print, print_todos, print_exchange, print_activity
from langchain_core.tools import tool
from tavily import TavilyClient
from deepagents import create_deep_agent

model = "claude-haiku-4-5-20251001"
research_system_prompt = """You are an expert research assistant."""

agent = create_deep_agent(
    model=model,
    system_prompt=research_system_prompt,
)

result = agent.invoke({"messages": [{"role": "user", "content": "What tools do you have?"}]})
pretty_print(result)

### Human

What tools do you have?

### Ai

I have access to the following tools:

1. **write_todos** — Create and manage structured task lists to track progress on complex, multi-step work.

2. **ls** — List files in a directory (requires absolute path).

3. **read_file** — Read file contents from the filesystem, with pagination support for large files.

4. **write_file** — Create new files in the filesystem.

5. **edit_file** — Edit existing files with exact string replacements.

6. **glob** — Find files matching glob patterns (e.g., `**/*.py`, `*.txt`).

7. **grep** — Search for text patterns across files, with options to show matches, count them, or list files.

8. **task** — Launch subagents to handle complex, isolated, multi-step tasks in parallel. Available subagent type: `general-purpose`.

These tools let me search, read, write, and edit files; manage task lists; and delegate complex work to specialized subagents. What would you like to do?

# The TODO list in action

#### Ask a deep agent to perform a complex task and it starts by planning out all the steps.

In [3]:
result = agent.invoke({"messages": [{"role": "user", "content": "I need to bake a cake. Make a plan"}]})
print_todos(result)

### Todo list

- ☐ Gather ingredients (flour, sugar, eggs, butter, baking powder, salt, vanilla, milk) — *pending*
- ☐ Preheat oven to recipe temperature (typically 350°F/175°C) — *pending*
- ☐ Prepare cake pans (grease and flour or line with parchment) — *pending*
- ☐ Mix dry ingredients (flour, baking powder, salt) — *pending*
- ☐ Cream butter and sugar together until light and fluffy — *pending*
- ☐ Add eggs one at a time, beating after each addition — *pending*
- ☐ Add vanilla extract — *pending*
- ☐ Alternate adding dry ingredients and milk to wet mixture — *pending*
- ☐ Pour batter into prepared pans — *pending*
- ☐ Bake for 25-35 minutes until toothpick comes out clean — *pending*
- ☐ Cool cakes in pans for 10-15 minutes — *pending*
- ☐ Turn out cakes onto wire racks to cool completely — *pending*
- ☐ Prepare frosting (if desired) — *pending*
- ☐ Decorate cake with frosting and toppings — *pending*

# Tools

#### Deep agents, as well as their subagents, can be extended with tools. Let's make ours more useful by allowing it to search the internet.

In [4]:
@tool
def search(query: str) -> str:
    """Search the internet for information

    Args:
        query (str): search query
    """
    results = TavilyClient().search(query)
    return results.get("results", [])

agent = create_deep_agent(
    model=model,
    system_prompt=research_system_prompt,
    tools=[search]
)

result = agent.invoke({"messages": [{"role": "user", "content": "What are the top 2 stories in the news today?"}]})
pretty_print(result)

### Human

What are the top 2 stories in the news today?

### Ai

I'll search for today's top news stories for you.

🔧 **search**(`{'query': 'top news today'}`)

### Tool

[{"url": "https://www.cbsnews.com", "title": "CBS News | Breaking news, top stories & today's latest headlines", "content": "homebuying JPMorgan Chase said the investment will include financing for 1 million affordable housing units and assistance to 500,000 people in buying homes.  Aug 3More [...] Image 77: SpaceX Reports First Quarterly Earnings Since Going Public  #### SpaceX shows strong growth in its first earnings report since IPO Elon Musk's rocket, satellite and AI provider reported quarterly revenue of $7.8 billion, topping Wall Street forecasts.  6H agoImage 78:  #### Top AI execs may meet with Trump officials Top executives from leading artificial intelligence companies are set to meet with White House officials, according to media reports. Ian Krietzberg, an AI correspondent [...] the decision was her own and \"was made from a thoughtful and empowered place.\"  20H ago  0:56Image 74:  #### Jordan Roth talks \"The Shards\" Tony Award-winning producer Jordan Roth talks about starring in the \"The Shards\" and describes what it was like to go from behind the scenes on Broadway to making his major on-screen acting debut.  19H ago  5:01Image 75:  #### Hollywood on hot streak at the box office \"Spider-Man: Brand New Day\" leapt straight to the top of the box office this weekend. Its", "score": 0.61140573, "raw_content": null, "id": "5578d1-00"}, {"url": "https://www.usnews.com", "title": "U.S. News & World Report: News, Rankings and Analysis on ...", "content": "TOPSHOT - Members of the Spanish Army's 'Regulares' infantry forces watch a group of migrants near the border post of the Spanish enclave of Ceuta on August 2, 2026. At least 72 people died during last week's mass influx of migrants into Spain's Ceuta enclave, when tens of thousands of people mostly swam around the border post from Morocco, local officials said today. Most of the estimated 60,000 people who rushed into Spain's north African territory have returned to Morocco since the [...] National News\n\n##### Today’s News: Michigan and the Moon\n\nBy Jenna Romaine\n\n### Data of the Day\n\n$1.2B\n\nin health product sales purchased through TikTok in the last 12 months. Learn which brands are leading the market.\n\nUnderstanding the fund's true purpose and guideline constraints are often overlooked in the diligence process.\n\nChief investment officer at Coastal Bridge Advisors\n\n## More Stories\n\n### Best Car Lease Deals\n\n### The 10 Worst Presidents\n\n### Doctor Finder Data and Methodologies [...] July 27, 2026, at 3:00 p.m.\n\n### More From U.S. News\n\n##### TRAVEL\n\n#### Celebrity vs. Virgin Voyages\n\nEastern vs. Western Caribbean Cruises\n\nWhat to Expect on an India River Cruise\n\nDoes Travel Insurance Cover Car Rentals?\n\n##### MONEY\n\n#### Student Loan Program Changes to Know\n\nFree Credit Monitoring Tool\n\nApple Card Credit Requirements\n\nThese Cards Help You Get Airline Status\n\n##### AUTOS\n\n#### Average Used Car Loan Interest Rates\n\nAverage Auto Loan Rates\n\n2027 Audi Q9 & SQ9 Preview", "score": 0.5150107, "raw_content": null, "id": "f82c85-01"}, {"url": "https://apnews.com", "title": "Associated Press News: Breaking News, Latest Headlines ...", "content": "TOP STORIES \n       Capybaras keep their chill while crashing Brazilian legislature\n       Fatou, the world's oldest gorilla living in captivity, celebrates her 69th birthday at Berlin Zoo\n       Raccoon goes on drunken rampage in Virginia liquor store and passes out on bathroom floor\n       Viral phenomenon in Argentina has young people identifying themselves as animals\n       Chicken wings advertised as 'boneless' can have bones, Ohio Supreme Court decides [...] Business\n\nSECTIONS TariffsInflationFinancial MarketsFinancial WellnessTechnology  \n\nTOP STORIES \n       SpaceX posts loss in first report as a public company but less than expected\n       Chipotle pulls jalapeños from some restaurants as health officials investigate salmonella outbreak\n\n   Science  \n   Newsletters  \n   Quizzes  \n   Games  \n   \nFact Check\n\nTOP STORIES \n       FACT FOCUS: No, you won't need an ID to shop at city-owned grocery stores in New York City\n\n   \nOddities [...] TOP STORIES \n       Rodrigo Paz ajusta su gabinete en medio de tensiones internas antes de anunciar reformas\n       El agua del Danubio baja tanto que emergen barcos de la Segunda Guerra Mundial\n       El Volcán de Fuego de Guatemala arroja lava y ceniza y provoca evacuaciones", "score": 0.5114976, "raw_content": null, "id": "70c380-02"}, {"url": "https://www.nytimes.com", "title": "The New York Times - Breaking News, US News, World News ...", "content": "## Top Stories\n\nWhite House Whipsaws Silicon Valley (and Itself) Over A.I. Rules\n\nThe Trump administration has struggled over how to approach “open source” models, which are freely available to download and favored by Chinese companies.\n\n6 min read\n\nLIVE\n\nAug. 4, 2026, 10:29 a.m. ET\n\nSenate Committee to Vote on Advancing Blanche Nomination\n\nTrump Meets With Pirro After Saying She ‘Choked’ in Reflecting Pool Case\n\n3 min read\n\nAgence France-Presse — Getty Images\n\nThe HeadlinesAudio", "score": 0.45420644, "raw_content": null, "id": "991311-03"}, {"url": "https://www.youtube.com/watch?v=JM3Kvs6sXwo", "title": "Morning News NOW Full Episode – Aug. 5", "content": "THE SUSPECT AND THE CONCERNING ITEMS POLICE FOUND IN HIS POSSESSION. MORE PRODUCE PROBLEMS AS THE CDC CONFIRMS CASES OF CYCLOSPORA INFECTIONS HAVE SURPASSED 10,000 ACROSS THE COUNTRY. ANOTHER BACTERIA CONNECTED TO FOOD POISONING IS ON THE RISE. THE POPULAR FAST CASUAL CHAIN OFFICIALS ARE EYEING AS THE POSSIBLE SOURCE OF A SALMONELLA SURGE, AND WHAT CONSUMERS NEED TO KNOW. LATER THIS HOUR. AND FINALLY, IT MAY BE TIME TO RETHINK YOUR TOP EIGHT. MYSPACE COULD BE MAKING A COMEBACK. OWNERS OF THE [...] WITH SOME WORRYING HEALTH NEWS AS OFFICIALS TRACK TWO MAJOR FOODBORNE ILLNESS OUTBREAKS. THERE ARE NOW MORE THAN 10,000 CONFIRMED CASES OF CYCLOSPORIASIS, WHILE SEPARATELY, MORE THAN 100 CASES OF SALMONELLA HAVE BEEN IDENTIFIED IN MINNESOTA. NBC NEWS CHIEF ENVIRONMENTAL AFFAIRS CORRESPONDENT ANNE THOMPSON HAS THE LATEST. AND GOOD MORNING. HEY THERE. FIRST IT WAS LETTUCE. NOW IT'S JALAPENOS. MORE THAN 100 CASES IN MINNESOTA, WITH A VAST MAJORITY OF THOSE PEOPLE HAVING EATEN RECENTLY AT CHIPOTLE, [...] THAT WE'VE BEEN DEALING WITH AGAIN, FOR THE EXTENDED PERIOD. WE'VE GOT THAT UP ACROSS PARTS OF THE WEST, WITH TEMPERATURES SURGING YET AGAIN INTO THE TRIPLE DIGITS, ESPECIALLY ACROSS THE SOUTHWEST TODAY, 111 IN PALM SPRINGS, BAKERSFIELD AND FRESNO, YOU TOP OUT AT 100 PLUS 112 FOR LAS VEGAS. THESE NUMBERS ARE ABOVE WHERE THEY SHOULD BE FOR THIS TIME OF YEAR. OF COURSE, IT IS SUMMER, BUT STILL 107 IN MEDFORD TODAY. THAT'S 16 DEGREES ABOVE OUR AVERAGE TEMPERATURE FOR THIS TIME OF YEAR. WE'VE GOT", "score": 0.428784, "raw_content": null, "id": "99296d-04"}]

### Ai

Based on today's news, here are the top 2 stories:

1. **SpaceX Earnings Report** - SpaceX reported its first quarterly earnings since going public, posting $7.8 billion in revenue, which exceeded Wall Street expectations. However, the company posted a loss in its first report as a public company, though it was less than expected.

2. **Chipotle Salmonella Outbreak** - Health officials are investigating a salmonella outbreak linked to jalapeños served at Chipotle restaurants. More than 100 cases have been identified in Minnesota, with the majority of those affected having recently eaten at Chipotle. This is part of a broader period of foodborne illness outbreaks, as over 10,000 cases of cyclospora have also been confirmed across the country.

# Task Delegation: Where Deep Agents Shine

#### The typical pattern is one main supervisor agent delegating to one or more specialized subagents.

### Each subagent has:
- its own context window
- its own tools
- its own system prompt
- its own model (can override the main agent's model)
- its own skills

### The main agent
- delegates via its task() tool
- sees only the final results of its subagents, keeping its context window clean

![Deep agent task delegation](images/deep-agent-delegation.svg)

# Defining Subagents: Dictionary Approach

#### Just define your subagent as a Python **dictionary** and plug it into your supervisor main agent.

In [5]:
research_subagent = {
    "name": "research-agent",
    "description": "Research a given task",
    "system_prompt": f"""You are a research assistant.
                         Use tools to gather information.
                         Structure findings with clear headings and inline citations.
                         Limit to 3 search calls.""",
    "model": "anthropic:claude-haiku-4-5-20251001",
    "tools": [search]
}

agent = create_deep_agent(model=model, system_prompt=research_system_prompt, subagents=[research_subagent])

# Defining Subagents: Compiled Approach

For more complex workflows we can use **compiled** subagents, allowing us arbitrary complexity.

In [6]:
from langchain.agents import create_agent
from deepagents import CompiledSubAgent

research_subagent = create_agent(
    model=model,
    tools=[search],
    system_prompt="""You are a research assistant.
                     Use the search tool to gather information.
                     Structure findings with clear headings and inline citations.
                     Limit to 3 search calls.""",
)

compiled_subagent = CompiledSubAgent(
    name="research-agent", description="Research a given task using web search", runnable=research_subagent,
)

agent = create_deep_agent(model=model, system_prompt=research_system_prompt, subagents=[compiled_subagent])

# Filesystems and backends

#### Deep agents are designed to handle long-running, complex tasks. They typically do this by decomposing tasks into sub-tasks and delegating execution to subagents. **Backends** expose a **filesystem** surface to allow (sub)agents to read and write state through a collection of tools: `ls, read_file, write_file, edit_file, glob, and grep`.

A few prebuilt filesystem backends that you can quickly use with your deep agent:

<table style="font-size:18px; border-collapse:collapse; width:100%;">
<thead>
<tr>
<th style="border:1px solid #999; padding:12px; text-align:left;">Backend</th>
<th style="border:1px solid #999; padding:12px; text-align:left;">Description</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Default<br><code style="font-size:15px; color:#2b8a3e;">create_deep_agent(model="google_genai:gemini-3.5-flash")</code></td>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Thread-scoped. The default filesystem backend for an agent is stored in langgraph state. Files persist across turns within a thread (via your checkpointer) and are not shared across threads.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Local filesystem persistence<br><code style="font-size:15px; color:#2b8a3e;">create_deep_agent(model="google_genai:gemini-3.5-flash", backend=FilesystemBackend(root_dir="/Users/nh/Desktop/"))</code></td>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Gives the deep agent access to your local machine's filesystem. You can specify the root directory that the agent has access to. Any provided <code style="font-size:15px; color:#2b8a3e;">root_dir</code> must be an absolute path. Typically, wrap in a <code style="font-size:15px; color:#2b8a3e;">CompositeBackend</code> to keep internal agent data (offloaded tool results, conversation history) separate from your project files.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Durable store (LangGraph store)<br><code style="font-size:15px; color:#2b8a3e;">create_deep_agent(model="google_genai:gemini-3.5-flash", backend=StoreBackend())</code></td>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Gives the agent access to long-term storage that is persisted across threads. Great for storing longer term memories or instructions that are applicable to the agent over multiple executions.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Context Hub<br><code style="font-size:15px; color:#2b8a3e;">create_deep_agent(model="google_genai:gemini-3.5-flash", backend=ContextHubBackend("my-agent"))</code></td>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Stores files durably in a LangSmith Hub repo, without provisioning a separate LangGraph store.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Sandbox<br><code style="font-size:15px; color:#2b8a3e;">create_deep_agent(model="google_genai:gemini-3.5-flash", backend=sandbox)</code></td>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Execute code in isolated environments. Sandboxes provide filesystem tools plus the <code style="font-size:15px; color:#2b8a3e;">execute</code> tool for running shell commands. Choose from LangSmith, AgentCore, Daytona, Deno, E2B, Modal, Runloop, or local VFS.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Local shell<br><code style="font-size:15px; color:#2b8a3e;">create_deep_agent(model="google_genai:gemini-3.5-flash", backend=LocalShellBackend(root_dir=".", env={"PATH": "/usr/bin:/bin"}))</code></td>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Filesystem and shell execution directly on the host. No isolation—use only in controlled development environments. See security considerations below.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Composite</td>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Thread-scoped by default, <code style="font-size:15px; color:#2b8a3e;">/memories/</code> persisted across threads. The Composite backend is maximally flexible. You can specify different routes in the filesystem to point towards different backends.</td>
</tr>
</tbody>
</table>

# Default Backend
- Virtual and ephemeral filesystem
- Default backend is thread-scoped
- Without a `thread_id` the agent cannot find the referenced filesystem

####  Thread ID is how we tell our agent that our response is part of a larger conversation.

In [7]:
from langgraph.checkpoint.memory import MemorySaver

agent = create_deep_agent(
    model=model, system_prompt=research_system_prompt, checkpointer=MemorySaver(),  # We don't have to mention the backend when using the 'default'; it is just available to us
)

config = {"configurable": {"thread_id": "joke-session"}}

result = agent.invoke({
    "messages": [{"role": "user", "content": "Tell me a joke about computer scientists and write it to /joke.md"}]
}, config=config)
pretty_print(result)

### Human

Tell me a joke about computer scientists and write it to /joke.md

### Ai

🔧 **write_file**(`{'file_path': '/joke.md', 'content': '# A Joke About Computer Scientists\n\nWhy do computer scientists make terrible lovers?\n\nBecause they always want to optimize for performance instead of enjoying the moment!\n\n---\n\n*Alternative version:*\n\nA computer scientist\'s wife tells him: "Go to the store and buy a loaf of bread. If they have eggs, buy a dozen."\n\nHe comes back with 12 loaves of bread.\n'}`)

### Tool

Updated file /joke.md

### Ai

Done! I've written a couple of computer scientist jokes to `/joke.md`. The first one's about their obsession with optimization, and the second is a classic about misinterpreting conditional logic.

#### We are able to read the filesystem because we pass in the same `config` object has the `thread_id` we previously used.

In [8]:
result = agent.invoke({
    "messages": [{"role": "user", "content": "Read /joke.md and tell me the joke"}]
}, config=config)
print_exchange(result)

### Human

Tell me a joke about computer scientists and write it to /joke.md

### Ai

Here are the jokes from the file:

**Main joke:**
Why do computer scientists make terrible lovers? Because they always want to optimize for performance instead of enjoying the moment!

**Alternative version:**
A computer scientist's wife tells him: "Go to the store and buy a loaf of bread. If they have eggs, buy a dozen."

He comes back with 12 loaves of bread.

(The last one's funny because he took the conditional literally — if eggs exist, he buys a dozen of something, which turns out to be the bread!)

#### This time the agent cannot find the file because we use a different `thread_id`

In [9]:
other_config = {"configurable": {"thread_id": "other-session"}}

result = agent.invoke({
    "messages": [{"role": "user", "content": "Read /joke.md and tell me the joke"}]
}, config=other_config)
print_exchange(result)

### Human

Read /joke.md and tell me the joke

### Ai

The root directory is empty, so `/joke.md` doesn't exist. Could you verify the file path or check if it's in a subdirectory?

# Store Backend
- Long-term
- Backed by a database
- Persisted across threads
- Good for storing longer term memories or instructions that are applicable to the agent over multiple executions.

#### In this example we write a memory in a thread named 'session-a'

In [10]:
from deepagents.backends import StoreBackend
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

agent = create_deep_agent(
    model=model,
    system_prompt=research_system_prompt,
    backend=StoreBackend(namespace=lambda ctx: ("memories",)),
    store=store,
)

# Write a memory in one thread...
session_a_config = {"configurable": {"thread_id": "session-a"}}

result = agent.invoke({
    "messages": [{"role": "user", "content": "Remember that my name is Andrew by writing it to /profile.md"}]
}, config=session_a_config)

pretty_print(result)

### Human

Remember that my name is Andrew by writing it to /profile.md

### Ai

🔧 **write_file**(`{'file_path': '/profile.md', 'content': '# Profile\n\nName: Andrew\n'}`)

### Tool

Updated file /profile.md

### Ai

Done. I've written your name to /profile.md.

#### Now we can read the memory back from a _different_ thread. Unlike the default backend, the **StoreBackend** persists files across threads.

In [11]:
session_b_config = {"configurable": {"thread_id": "session-b"}}

result = agent.invoke({
    "messages": [{"role": "user", "content": "Read /profile.md and tell me my name"}]
}, config=session_b_config)

print_exchange(result)

### Human

Read /profile.md and tell me my name

### Ai

Your name is Andrew.

# Store Backend: Choosing the backing database

The <code style="font-size:15px; color:#2b8a3e;">StoreBackend</code> is <b>storage-agnostic</b>. It is a thin filesystem adapter over whatever LangGraph <code style="font-size:15px; color:#2b8a3e;">BaseStore</code> you pass to <code style="font-size:15px; color:#2b8a3e;">create_deep_agent(store=...)</code>. The database is chosen entirely by that one argument &mdash; the agent code never changes.

<pre style="background:#f6f8fa; border:1px solid #d0d7de; border-radius:6px; padding:12px; font-size:14px; line-height:1.5; overflow:auto;"><code>agent = create_deep_agent(
    model=model,
    backend=StoreBackend(namespace=lambda ctx: ("memories",)),
    store=<span style="color:#cf222e;">&lt;ANY BaseStore&gt;</span>,   <span style="color:#6e7781;"># &larr; this is the database choice</span>
)</code></pre>

<table style="font-size:16px; border-collapse:collapse; width:100%;">
<thead>
<tr>
<th style="border:1px solid #999; padding:10px; text-align:left;">Store</th>
<th style="border:1px solid #999; padding:10px; text-align:left;">Package</th>
<th style="border:1px solid #999; padding:10px; text-align:left;">Import</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left;"><code style="color:#2b8a3e;">InMemoryStore</code> <span style="color:#6e7781;">(dev only)</span></td>
<td style="border:1px solid #999; padding:10px; text-align:left;">built-in</td>
<td style="border:1px solid #999; padding:10px; text-align:left;"><code style="color:#2b8a3e;">langgraph.store.memory</code></td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left;"><code style="color:#2b8a3e;">PostgresStore</code></td>
<td style="border:1px solid #999; padding:10px; text-align:left;"><code style="color:#2b8a3e;">langgraph-checkpoint-postgres</code></td>
<td style="border:1px solid #999; padding:10px; text-align:left;"><code style="color:#2b8a3e;">langgraph.store.postgres</code></td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left;"><code style="color:#2b8a3e;">RedisStore</code></td>
<td style="border:1px solid #999; padding:10px; text-align:left;"><code style="color:#2b8a3e;">langgraph-checkpoint-redis</code></td>
<td style="border:1px solid #999; padding:10px; text-align:left;"><code style="color:#2b8a3e;">langgraph.store.redis</code></td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left;"><b><code style="color:#2b8a3e;">MongoDBStore</code></b></td>
<td style="border:1px solid #999; padding:10px; text-align:left;"><b><code style="color:#2b8a3e;">langgraph-store-mongodb</code></b></td>
<td style="border:1px solid #999; padding:10px; text-align:left;"><b><code style="color:#2b8a3e;">langgraph.store.mongodb</code></b></td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left;">Anything custom</td>
<td style="border:1px solid #999; padding:10px; text-align:left;">your code</td>
<td style="border:1px solid #999; padding:10px; text-align:left;">subclass <code style="color:#2b8a3e;">BaseStore</code></td>
</tr>
</tbody>
</table>

### MongoDB example

<pre style="background:#f6f8fa; border:1px solid #d0d7de; border-radius:6px; padding:12px; font-size:14px; line-height:1.5; overflow:auto;"><code><span style="color:#cf222e;">from</span> langgraph.store.mongodb <span style="color:#cf222e;">import</span> MongoDBStore

<span style="color:#cf222e;">with</span> MongoDBStore.from_conn_string(
    conn_string=<span style="color:#0a3069;">"mongodb://localhost:27017"</span>,
    db_name=<span style="color:#0a3069;">"deepagents"</span>,
    collection_name=<span style="color:#0a3069;">"memories"</span>,
) <span style="color:#cf222e;">as</span> store:
    agent = create_deep_agent(
        model=model,
        system_prompt=research_system_prompt,
        backend=StoreBackend(namespace=<span style="color:#cf222e;">lambda</span> ctx: (<span style="color:#0a3069;">"memories"</span>,)),
        store=store,   <span style="color:#6e7781;"># only this line changed &mdash; files now persist in MongoDB</span>
    )
    <span style="color:#6e7781;"># ...same invoke() calls as the InMemoryStore cell</span></code></pre>


# Composite Backend
- Flexible: specify different routes in the filesystem to point towards different backends.

#### Example:
- Use <code style="font-size:15px; color:#2b8a3e;">StateBackend</code> for thread-scoped default storage
- Save persistent memories to <code style="font-size:15px; color:#2b8a3e;">/memories/</code>. These files are available from different threads.
- Save ephemeral memories to the root <code style="font-size:15px; color:#2b8a3e;">/</code>. These are not available across different threads.



In [12]:
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

# Route by path: everything is ephemeral (StateBackend) except /memories/, which is routed to the durable StoreBackend.
composite_backend = CompositeBackend(
    default=StateBackend(),
    routes={"/memories/": StoreBackend(namespace=lambda ctx: ("memories",))},
)

agent = create_deep_agent(
    model=model, system_prompt=research_system_prompt, backend=composite_backend, store=store,
)

# Thread A: write one ephemeral file and one durable file
result = agent.invoke({
    "messages": [{"role": "user", "content": (
        "Write a quick scratch note to /draft.md, and save that my name is Andrew to /memories/profile.md"
    )}]
}, config={"configurable": {"thread_id": "thread-a"}})
pretty_print(result)

### Human

Write a quick scratch note to /draft.md, and save that my name is Andrew to /memories/profile.md

### Ai

🔧 **write_file**(`{'file_path': '/draft.md', 'content': 'Quick scratch note\n\nTODO:\n- [ ] Add more details here\n'}`)

🔧 **write_file**(`{'file_path': '/memories/profile.md', 'content': 'name: Andrew\n'}`)

### Tool

Updated file /draft.md

### Tool

Updated file /memories/profile.md

### Ai

Done. Created `/draft.md` with a scratch note template and saved your name to `/memories/profile.md`.

In [13]:
# Thread B: a *different* thread. The /memories/ route persisted; /draft.md did not.
result = agent.invoke({
    "messages": [{"role": "user", "content": (
        "Try to read both /memories/profile.md and /draft.md. Report exactly what you find for each."
    )}]
}, config={"configurable": {"thread_id": "thread-b"}})
print_exchange(result)

### Human

Try to read both /memories/profile.md and /draft.md. Report exactly what you find for each.

### Ai

**Results:**

- **/memories/profile.md**: Contains a single line: `name: Andrew`

- **/draft.md**: File not found (does not exist)

# Context Isolation vs. Shared Filesystems

When a supervisor delegates to a subagent, two different things happen to two different kinds of state:

<table style="font-size:16px; border-collapse:collapse; width:100%;">
<thead>
<tr>
<th style="border:1px solid #999; padding:10px; text-align:left; width:22%;">What</th>
<th style="border:1px solid #999; padding:10px; text-align:left;">Behavior</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><b>Message context</b><br><span style="color:#6e7781;">(isolated)</span></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">The subagent runs in its <b>own context window</b>. Its intermediate reasoning and tool calls never enter the supervisor's history &mdash; only a single, compact <b>final report</b> is returned. This is the whole point of subagents: keep the supervisor's context clean.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><b>Filesystem</b><br><span style="color:#6e7781;">(shared)</span></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">The backend is <b>shared</b> between the supervisor and all subagents. Any file a subagent writes <b>remains in state after the subagent finishes</b> and stays readable by the supervisor and sibling subagents.</td>
</tr>
</tbody>
</table>

### The pattern this unlocks

The filesystem is a **shared side-channel** for moving large results across the subagent boundary *without* paying for them in context:

1. Supervisor delegates: *"research X and write the findings to <code style="color:#2b8a3e;">/research.md</code>"*
2. Subagent does the heavy work, writes the file, and returns just *"wrote results to /research.md"*
3. Supervisor (or another subagent) reads <code style="color:#2b8a3e;">/research.md</code> when it needs the detail

The bulky output travels through the filesystem; only a one-line handoff travels through the conversation.

<b>Caveat:</b> dictionary subagents inherit the supervisor's backend automatically. A fully custom <code style="font-size:15px; color:#2b8a3e;">CompiledSubAgent</code> graph has its own state and only shares the filesystem if you wire it to the same backend.

In [14]:
from langgraph.checkpoint.memory import MemorySaver

# A subagent whose job is to write its findings to a file (not to report them inline).
writer_subagent = {
    "name": "writer-agent",
    "description": "Writes the requested content to a file, then reports only the path.",
    "system_prompt": (
        "Write the requested content to the requested file path using write_file. "
        "Then report ONLY the path you wrote to — do not repeat the content."
    ),
    "tools": [],
}

agent = create_deep_agent(
    model=model,
    system_prompt=(
        "You are a supervisor. Delegate writing tasks to the writer-agent subagent via the task tool. "
        "The subagent shares your filesystem, so after it finishes you can read the file it wrote yourself."
    ),
    subagents=[writer_subagent],
    checkpointer=MemorySaver(),
)

config = {"configurable": {"thread_id": "subagent-fs-demo"}}

# print_activity streams the run with subgraphs=True, so we can watch the *subagent*
# call write_file inside its own isolated context — that tool call never appears in
# the supervisor's returned messages.
print_activity(agent, (
    "Delegate to writer-agent: have it write a haiku about the ocean to /poem.md. "
    "After the subagent finishes, YOU read /poem.md with read_file and quote its exact contents back to me."
), config=config)

### Activity timeline

- 🧑‍✈️ **supervisor** → 📨 delegates via **task** to `writer-agent`
    - 🤖 **writer-agent** → 🔧 **write_file**(`/poem.md`, content=“# Ocean Haiku Waves crash on the shore Endless blue horizon calls Salt wind touc…”)
    - 🤖 **writer-agent** ← 📥 `write_file` → Updated file /poem.md
- 🧑‍✈️ **supervisor** ← 📥 `task` → /poem.md
- 🧑‍✈️ **supervisor** → 🔧 **read_file**(`/poem.md`)
- 🧑‍✈️ **supervisor** ← 📥 `read_file` → 1 # Ocean Haiku 2 3 Waves crash on the shore 4 Endless blue horizon calls 5 Salt…

#### Because we passed a checkpointer + thread_id, the run's state is durable on this thread. The subagent's file lives in the shared filesystem, readable by the main supervisor agent.

In [15]:
state = agent.get_state(config)
print("Files in shared state:", list(state.values.get("files", {}).keys()))

# And the supervisor's own transcript only ever saw the delegation + the final read —
# never the subagent's write_file call (that stayed isolated in the subagent's context).
pretty_print(state.values)

Files in shared state: ['/poem.md']


### Human

Delegate to writer-agent: have it write a haiku about the ocean to /poem.md. After the subagent finishes, YOU read /poem.md with read_file and quote its exact contents back to me.

### Ai

🔧 **task**(`{'subagent_type': 'writer-agent', 'description': 'Write a haiku about the ocean to the file /poem.md. The haiku should follow the traditional 5-7-5 syllable structure. After writing, report the file path.'}`)

### Tool

/poem.md

### Ai

Now let me read the file:

🔧 **read_file**(`{'file_path': '/poem.md'}`)

### Tool

     1	# Ocean Haiku
     2	
     3	Waves crash on the shore
     4	Endless blue horizon calls
     5	Salt wind touches face

### Ai

Here's the exact contents of /poem.md:

```
# Ocean Haiku

Waves crash on the shore
Endless blue horizon calls
Salt wind touches face
```

# Context Management Techniques

### A. Subagent Context Isolation

Each subagent gets its own, isolated context window. This ensures that context relevant to a specific task is only seen by the subagent working on that task.

### B. Offloading Large Inputs and Outputs

When a tool returns (or receives) more than comfortably fits in context, the harness writes the payload to the filesystem and replaces it in the message history with a short **reference plus a preview** (e.g. the first 10 lines). The agent calls <code style="color:#2b8a3e;">read_file</code> to pull back more **only if it actually needs the detail** — instead of dragging thousands of lines through every subsequent turn.

<img src="images/offloading-results.png" alt="Offloading large tool results to the filesystem" style="max-width:100%; width:820px; border:1px solid #d0d7de; border-radius:6px;" />

<ul style="font-size:16px;">
<li>Oversized tool results are evicted and swapped for a reference.</li>
<li>The agent reads back just the slice it needs via <code style="color:#2b8a3e;">read_file</code>.</li>
</ul>

### C. Context Summarization

When the **conversation itself** approaches the model's limit (~85% of <code style="color:#2b8a3e;">max_input_tokens</code>), <code style="color:#2b8a3e;">SummarizationMiddleware</code> runs a **compaction step**: older turns are compressed into an LLM-generated summary — session intent, artifacts created, next steps — while the most recent ~10% of tokens are kept verbatim.

<ul style="font-size:16px;">
<li><b>In-context summary + pointer:</b> the summary replaces the old messages and tells the agent where the full record lives.</li>
<li><b>Filesystem preservation:</b> the original messages are written to <code style="color:#2b8a3e;">/conversation_history/{thread_id}.md</code>, so the agent can search or re-read them if needed.</li>
</ul>

<img src="images/summarization.png" alt="Summarizing older conversation turns into a compact summary + pointer" style="max-width:100%; width:820px; border:1px solid #d0d7de; border-radius:6px;" />

